In [55]:
import torch 
import torch.nn as nn
import tiktoken
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

In [56]:
class LayerNorm(nn.Module):
    def __init__(self,embed_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(embed_dim))
        self.shift = nn.Parameter(torch.zeros(embed_dim))

    def forward(self,x):
        mean = x.mean(dim = -1,keepdim = True)
        var = x.var(dim = -1,keepdim = True,unbiased = True)
#       Biased	Divide by N	Usually used when you have the entire population.
#       Unbiased	Divide by N−1	Used when estimating the population variance from a sample.
#       To correct this "underestimation," we divide by a smaller number (n−1 instead of n). Mathematically, dividing by a smaller 
#       denominator results in a slightly larger variance.

        norm_x = (x-mean) / torch.sqrt(var + self.eps)
        return self.scale * norm_x + self.shift

In [57]:
class GELU(nn.Module):
    def __init__(self):
        super().__init__()
        self.register_buffer('c',torch.sqrt(torch.tensor(2.0 / torch.pi)))
    def forward(self,x):
        return 0.5 * x * (1 + torch.tanh(self.c * (x + 0.044715 * x.pow(3))))
    

In [58]:
class MyLinear(nn.Module):
    def __init__(self,in_features,out_features):
        super().__init__()

        self.weights = nn.Parameter(torch.randn(out_features,in_features)*0.02)
        self.bias = nn.Parameter(torch.randn(out_features))

    def forward(self,x):
        return x @self.weights.t() + self.bias




class FeedForward(nn.Module):
    def __init__(self,cfg):
        super().__init__()
        self.layers = nn.Sequential(
            MyLinear(cfg["embed_dim"],4*cfg["embed_dim"]),
            GELU(),
            MyLinear(4*cfg["embed_dim"],cfg["embed_dim"])
            
        )
    def forward(self,x):
        return self.layers(x)

In [59]:
class MultiHeadAttention(nn.Module):
    def __init__(self,d_in,d_out,num_heads,dropout,context_len,qvk_bias = False):
        super().__init__()
        assert (d_out % num_heads == 0), "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads

        self.W_query = nn.Linear(d_in,d_out,bias=qvk_bias)
        self.W_value = nn.Linear(d_in,d_out,bias=qvk_bias)
        self.W_key = nn.Linear(d_in,d_out,bias=qvk_bias)
        self.project_out = nn.Linear(d_out,d_out)
        self.dropout = nn.Dropout(dropout)

    def forward(self,x):
        batch,num_tokens,d_in = x.shape

        keys = self.W_key(x)
        values = self.W_value(x)
        query = self.W_query(x)

        keys = keys.view(batch,num_tokens,self.num_heads,self.head_dim)
        values = values.view(batch,num_tokens,self.num_heads,self.head_dim)
        query = query.view(batch,num_tokens,self.num_heads,self.head_dim)

        keys = keys.transpose(1,2)
        values = values.transpose(1,2)
        query = query.transpose(1,2)

        attn_scores = query @ keys.transpose(-2,-1)

        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5,dim=-1)
        attn_weights = self.dropout(attn_weights)
        context_vec = (attn_weights @ values).transpose(1,2)

        context_vec = context_vec.contiguous().view(batch,num_tokens,self.d_out)

        context_vec = self.project_out(context_vec)

        return context_vec


In [60]:
class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.attn = MultiHeadAttention(
            d_in=cfg["embed_dim"],
            d_out=cfg["embed_dim"],
            num_heads=cfg["n_heads"],
            dropout=cfg["drop_rate"],
            context_len=cfg["context_length"],
            qvk_bias=cfg["qvk_bias"])
        self.ff = FeedForward(cfg)

        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])

        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

    def forward(self,x):
        shortcut = x
        x = self.attn(x)
        x = self.drop_shortcut(x)
        x = self.norm1(x + shortcut)

        shortcut = x
        x = self.ff(x)
        x = self.drop_shortcut(x)
        x = self.norm2(x + shortcut)

        return x

In [61]:
cfg = {
    "embed_dim": 768,
    "emb_dim": 768,        # used in LayerNorm
    "n_heads": 12,
    "drop_rate": 0.1,
    "context_length": 16,
    "qvk_bias": True
}


block = TransformerBlock(cfg)

In [62]:
print(block)

TransformerBlock(
  (attn): MultiHeadAttention(
    (W_query): Linear(in_features=768, out_features=768, bias=True)
    (W_value): Linear(in_features=768, out_features=768, bias=True)
    (W_key): Linear(in_features=768, out_features=768, bias=True)
    (project_out): Linear(in_features=768, out_features=768, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (ff): FeedForward(
    (layers): Sequential(
      (0): MyLinear()
      (1): GELU()
      (2): MyLinear()
    )
  )
  (norm1): LayerNorm()
  (norm2): LayerNorm()
  (drop_shortcut): Dropout(p=0.1, inplace=False)
)


In [63]:
x = torch.randn(2, 16, 768)   # [batch, tokens, embed_dim]

out = block(x)

print("Input shape :", x.shape)
print("Output shape:", out.shape)

Input shape : torch.Size([2, 16, 768])
Output shape: torch.Size([2, 16, 768])


In [64]:
class BERTEmbedding(nn.Module):
    def __init__(self,vocab_size,emb_dim,max_len,drop_rate):
        super().__init__()

        self.token_emb = nn.Embedding(vocab_size,emb_dim)
        self.pos_emb = nn.Embedding(max_len,emb_dim)
        self.segment_emb = nn.Embedding(2,emb_dim)

        self.norm = LayerNorm(emb_dim)
        self.dropout = nn.Dropout(drop_rate)

    def forward(self,input_ids,segment_ids):
        B,T = input_ids.shape
        positions = torch.arange(T,device = input_ids.device)
        positions = positions.unsqueeze(0).expand(B,T)

        token = self.token_emb(input_ids)
        pos = self.pos_emb(positions)
        seg = self.segment_emb(segment_ids)

        x = token + pos + seg

        x = self.norm(x)
        x = self.dropout(x)

        return x

    

In [65]:
class BERTEncoder(nn.Module):
    def __init__(self, cfg):
        super().__init__()

        self.layers = nn.ModuleList(
            [TransformerBlock(cfg) for _ in range(cfg["n_layers"])]
        )

    def forward(self, x):

        for layer in self.layers:
            x = layer(x)

        return x

In [66]:
class BERTPooler(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()

        self.linear = nn.Linear(emb_dim, emb_dim)
        self.activation = nn.Tanh()

    def forward(self, x):

        cls_token = x[:, 0]      # first token
        pooled = self.linear(cls_token)

        return self.activation(pooled)

In [67]:
class BERT(nn.Module):
    def __init__(self, cfg):
        super().__init__()

        self.embedding = BERTEmbedding(
            vocab_size=cfg["vocab_size"],
            emb_dim=cfg["emb_dim"],
            max_len=cfg["context_length"],
            drop_rate=cfg["drop_rate"]
        )

        self.encoder = BERTEncoder(cfg)

        self.pooler = BERTPooler(cfg["emb_dim"])

    def forward(self, input_ids, segment_ids):

        x = self.embedding(input_ids, segment_ids)

        x = self.encoder(x)

        pooled = self.pooler(x)

        return x, pooled

In [68]:
cfg = {
    "vocab_size": 30522,
    "emb_dim": 768,
    "embed_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate": 0.1,
    "context_length": 512,
    "qvk_bias": True
}
model = BERT(cfg)

input_ids = torch.randint(0, 30522, (2, 16))
segment_ids = torch.zeros_like(input_ids)

token_out, cls_out = model(input_ids, segment_ids)

print(token_out.shape)
print(cls_out.shape)

torch.Size([2, 16, 768])
torch.Size([2, 768])


In [69]:
import random

def create_sentence_pairs(sentences):
    pairs = []

    for i in range(len(sentences)-1):

        if random.random() < 0.5:
            pairs.append((sentences[i], sentences[i+1], 0))  # IsNext
        else:
            rand = random.choice(sentences)
            pairs.append((sentences[i], rand, 1))  # NotNext

    return pairs

In [70]:
def mask_tokens(input_ids, tokenizer):

    labels = input_ids.clone()

    probability = torch.rand(input_ids.shape)

    mask = probability < 0.15
    labels[~mask] = -100   # ignore index

    # 80% replace with [MASK]
    indices = mask & (torch.rand(input_ids.shape) < 0.8)
    input_ids[indices] = tokenizer.mask_token_id

    return input_ids, labels

In [71]:
def encode_pair(tokenizer, sent_a, sent_b, max_len):

    encoded = tokenizer(
        sent_a,
        sent_b,
        padding="max_length",
        truncation=True,
        max_length=max_len,
        return_tensors="pt"
    )

    return encoded

In [72]:
from torch.utils.data import Dataset

class BertPretrainDataset(Dataset):

    def __init__(self, sentences, tokenizer, max_len):

        self.pairs = create_sentence_pairs(sentences)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):

        sent_a, sent_b, label = self.pairs[idx]

        encoded = encode_pair(self.tokenizer, sent_a, sent_b, self.max_len)

        input_ids = encoded["input_ids"].squeeze(0)
        segment_ids = encoded["token_type_ids"].squeeze(0)

        input_ids, mlm_labels = mask_tokens(input_ids, self.tokenizer)

        return {
            "input_ids": input_ids,
            "segment_ids": segment_ids,
            "mlm_labels": mlm_labels,
            "nsp_labels": torch.tensor(label)
        }

In [73]:
sentences = [
    "The cat sat on the mat.",
    "It was very sleepy.",
    "Dogs like to play outside.",
    "The weather is sunny."
]
from torch.utils.data import DataLoader

dataset = BertPretrainDataset(sentences, tokenizer, max_len=32)

loader = DataLoader(dataset, batch_size=4, shuffle=True)

In [74]:
batch = next(iter(loader))

print(batch["input_ids"].shape)
print(batch["segment_ids"].shape)
print(batch["mlm_labels"].shape)
print(batch["nsp_labels"].shape)

torch.Size([3, 32])
torch.Size([3, 32])
torch.Size([3, 32])
torch.Size([3])
